## This is a brief demo of the 'new hardening' class `FixedOuterTime_InnerPL_SAM`

(More explanation to be added here. For now, see the docstring below.)

In [ ]:
%reload_ext autoreload
%autoreload 2

# standard secondary packages
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp


# development packages
import kalepy as kale
import kalepy.utils
import kalepy.plot

# --- Holodeck ----
import holodeck as holo
from holodeck import utils, plot
from holodeck.constants import YR, GYR, PC, MSOL, NWTG, SPLC
import holodeck.sams


## Docstring for `FixedOuterTime_InnerPL_SAM` (new hardening) class

In [ ]:
print(holo.hardening.FixedOuterTime_InnerPL_SAM.__doc__)

## A simple example

In [ ]:
sam = holo.sams.Semi_Analytic_Model()
hard = holo.hardening.FixedOuterTime_InnerPL_SAM(sam)

# check if hardening model params are physical, skip gwb calculation if not
if np.any(hard._params_allowed==False):
    print(f"Invalid hardening model! Skipping gwb calculation."
          f"  (rchar9={hard._rchar_9}pc, alpha_char={hard._alpha_char}, "
          f"  nu_inner={hard._nu_inner},\n"
          f"  r9={hard._r_gw_crit_9} [{hard._gw_crit_units}], "
          f"  alpha_gw={hard._alpha_gw_crit} beta_gw={hard._beta_gw_crit})")
    hc_ss, hc_bg = None, None

else:
    freqs, freqs_edges = utils.pta_freqs()
    hc_ss, hc_bg = sam.gwb(freqs_edges, hard=hard, realize=10)

    fig = plot.plot_gwb(freqs, hc_bg)

## More detailed examples

In [ ]:
def plot_gwb_and_dadt(nrads=100, grid_shape=None, 
                      gsmf_flag=2, gpf_flag=0,
                      tau_outer=1.0, nu_in=0.0, 
                      rch9=1.0, alpha_ch=-2.0/3,
                      gwc_units='rg', rgw9=10**2.5, 
                      alpha_gw=-0.25, beta_gw=0.25,
                      calc_gwb=False, nreals=10, nloud=1,
                      inner_mod_type=0,
                      enforce_phys_pars=False):

    # Are we using galaxy pair fraction + merger timescale
    # or are we using the Illustris galaxy merger rates?
    if gpf_flag: 
        _gpf = sams.GPF_Power_Law()
    else:
        _gpf = None

    # Are we using a single or double Schechter function 
    # for the galaxy stellar mass function?
    if gsmf_flag == 1:
        _gsmf = holo.sams.GSMF_Schechter()
    elif gsmf_flag == 2:
        _gsmf = holo.sams.GSMF_Double_Schechter()
    else: 
        raise ValueError(
            "keyword `gsmf_flag` must be 1 for single Schecter "
            "or 2 for double Schechter"
        )

    # create the SAM
    sam = holo.sams.Semi_Analytic_Model(
        shape = grid_shape,
        gsmf = _gsmf,
        gpf = _gpf
    )
    print(f"finished creating SAM class instance.")

    # create the new-hardening instance
    hard = holo.hardening.FixedOuterTime_InnerPL_SAM(
        sam, 
        num_steps = nrads,
        outer_time = tau_outer * GYR,
        rchar_9 = rch9 * PC,
        alpha_char = alpha_ch,
        nu_inner = nu_in,
        gw_crit_units = gwc_units,
        r_gw_crit_9 = rgw9, 
        alpha_gw_crit = alpha_gw, 
        beta_gw_crit = beta_gw, 
        inner_model_type=inner_mod_type,
        enforce_physical_params=enforce_phys_pars,
    )
    print(f"finished creating hardening class instance.")

    # check if hardening model params are physical, skip gwb calculation if not
    if np.any(hard._params_allowed==False):
        print(f"Invalid hardening model!"
              f"  (rchar9={hard._rchar_9}pc, alpha_char={hard._alpha_char}, "
              f"  nu_inner={hard._nu_inner},\n"
              f"  r9={hard._r_gw_crit_9} [{hard._gw_crit_units}], "
              f"  alpha_gw={hard._alpha_gw_crit} beta_gw={hard._beta_gw_crit})")
        if calc_gwb:
            print(f"Skipping gwb calculation b/c hard params invalid.")
        else:
            print(f"Skipping gwb calculation b/c {calc_gwb=} & hard params invalid.")            
    else:
        if calc_gwb:
            freqs, freqs_edges = utils.pta_freqs()
            hc_ss, hc_bg = sam.gwb(freqs_edges, hard=hard, realize=nreals)
            fig = plot.plot_gwb(freqs, hc_bg)
            print(f"finished gwb calculation.")            
        else:
            print(f"Skipping gwb calculation b/c {calc_gwb=}.")

    # ---- Define an array of binary separations and calculate dadt at each
    # start from the inner hardening model's initial separation (rchar)
    # (after the fixed-time 'outer' hardening phase has ended)
    m9 = sam.mtot / (1.0e9*MSOL)
    rchar = hard._rchar_9 * m9**(hard._alpha_char+1)
    rmax = rchar
    
    # (M,) end at the ISCO
    rmin = utils.rad_isco(sam.mtot)
    
    # Choose steps for each binary, log-spaced between rmin and rmax (M, X)
    extr = np.log10([rmax * np.ones_like(rmin), rmin])
    radii = np.linspace(0.0, 1.0, nrads)[np.newaxis, :]
    radii = extr[0][:, np.newaxis] + (extr[1] - extr[0])[:, np.newaxis] * radii
    radii = 10.0 ** radii
    
    # (M, Q, Z, X)
    mt, mr, rz, rads = np.broadcast_arrays(
        sam.mtot[:, np.newaxis, np.newaxis, np.newaxis],
        sam.mrat[np.newaxis, :, np.newaxis, np.newaxis],
        sam.redz[np.newaxis, np.newaxis, :, np.newaxis],
        radii[:, np.newaxis, np.newaxis, :]
    )

    # (M, Q, Z, X) is shape of input and output arrays
    dadt, agw_crit, rz_char, rz_final = hard.dadt(mt, mr, rz, rads)
    print(f"finished dadt calculation.")

    print(dadt.shape,rads.shape)
    time_evo = -utils.trapz_loglog(-1.0 / dadt[:,:,0,:], rads[:,:,0,:], 
                                   axis=2, cumsum=True)
    print(f"finished time_evo calculation.")

    if hard._gw_crit_units == 'pc':
        dunits = PC
        xlim=[1e-8,1e5]        
    else:
        dunits = NWTG * sam.mtot / SPLC**2
        xlim=[1,1e13]

    max_to_plot = 4
    if sam.mtot.size > max_to_plot:
        mt_nskip = int((sam.mtot.size-1)/(max_to_plot-1))        
    if sam.mrat.size > max_to_plot:
        mr_nskip = int((sam.mrat.size-1)/(max_to_plot-1))        

    cmap = plot._get_cmap('PuBuGn')
    colors = cmap(np.linspace(0.3, 1, max_to_plot+1))
    lw = np.arange(0.5,max_to_plot+1, 0.5)
    
    fig = plt.figure(figsize=(12,4))
    first_plot_index = 131
    kwargs = dict(xscale='log', yscale='log', xlim=xlim,
                  xlabel=f'binary separation [{hard._gw_crit_units}]')
    ax1 = fig.add_subplot(first_plot_index, **kwargs)
    ax1.xaxis.set_inverted(True)
    ax1.set_ylabel('hardening tscale [yr]')

    ax2 = fig.add_subplot(first_plot_index+1, **kwargs)
    ax2.xaxis.set_inverted(True)
    ax2.set_ylabel('hardening rate [cm/s]')

    ax3 = fig.add_subplot(first_plot_index+2, **kwargs)
    ax3.xaxis.set_inverted(True)
    ax3.set_ylabel('cumulative time [yr]')
    
    lw = np.arange(0.5,max_to_plot+1, 0.5)
        
    i_plot=0
    for i in np.arange(0,sam.mtot.size,mt_nskip):
            j_plot=0
            for j in np.arange(0,sam.mrat.size,mr_nskip):
                
                ax1.plot(rads[i,j,0,:]/dunits[i], -rads[i,j,0,:]/dadt[i,j,0,:]/YR, 
                         alpha=0.5, color=colors[i_plot], lw=lw[j_plot])
                lbl=f"mtot={sam.mtot[i]/MSOL:.2g}" if j_plot==max_to_plot-1 else None
                ax2.plot(rads[i,j,0,:]/dunits[i], -dadt[i,j,0,:], 
                         alpha=0.5, color=colors[i_plot], lw=lw[j_plot], 
                         label=lbl)
                ax3.plot(rads[i,j,0,:-1]/dunits[i], time_evo[i,j,:]/YR, 
                         alpha=0.5, color=colors[i_plot], lw=lw[j_plot])
                
                j_plot += 1
            i_plot += 1
        
    ax2.plot(xlim, [SPLC,SPLC], color='magenta')
    ax2.legend(fontsize=9)
    fig.subplots_adjust(wspace=0.3,top=0.9, right=0.95)
    

### Example hardening model with default parameters

In [ ]:
plot_gwb_and_dadt()

### Example hardening model in which agw has no mass or mass ratio dependence

In [ ]:
plot_gwb_and_dadt(alpha_gw=0, beta_gw=0)

### Example hardening model with 'stellar-like' parameters

In [ ]:
plot_gwb_and_dadt(nu_in=-1.0, rgw9=10**3.5)

### Example hardening model with 'gas-like' parameters

In [ ]:
plot_gwb_and_dadt(nu_in=2.0, rgw9=100.0)

### Example of a hardening model that has unphysical parameters

In [ ]:
plot_gwb_and_dadt(nu_in=-1.0, rch9=100)

### Another example of a hardening model that has unphysical parameters
#### This time we will force the hardening class to throw an error

In [ ]:
plot_gwb_and_dadt(nu_in=-0.5, rch9=10, alpha_gw=0, 
                  enforce_phys_pars=True)